# Governança e qualidade dos dados (Etapa 2)
Projeto de previsão de demanda de energia elétrica para apoio ao planejamento de capacidade.

Base usada, carga de energia diária do Portal de Dados Abertos do ONS.


## 1. Configuração

In [ ]:
import hashlib
import json
import platform
import urllib.request
from datetime import datetime, date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# PARÂMETROS DO PROJETO
ANO_INICIO_INVENTARIO = 2000      # baixa a base inteira para avaliar a qualidade de tudo
ANO_FIM = datetime.now().year     # último arquivo anual disponível
SUBSISTEMA_ALVO = "S"             # Sul (escopo do trabalho)
JANELA_INICIO = "2015-01-01"      # início do recorte usado no dataset tratado
MAX_LACUNA_IMPUTAVEL = 3          # lacunas de até N dias seguidos são interpoladas
LIMIAR_Z_ROBUSTO = 3.5            # limiar para marcar outliers
REBAIXAR = False                  # True força baixar de novo mesmo com cache local
USAR_INMET = False                # True ativa a seção opcional de temperatura

URL_ONS = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_{ano}.csv"
URL_PORTAL = "https://dados.ons.org.br/dataset/carga-energia"

# marcos de metodologia do ONS
MARCO_METODOLOGIA_1 = pd.Timestamp("2021-03-01")   # passa a incluir previsão de usinas não despachadas
MARCO_METODOLOGIA_2 = pd.Timestamp("2023-04-29")   # passa a incluir estimativa de MMGD

# eventos conhecidos para checar a acurácia
EVENTOS_CONHECIDOS = {
    "pandemia_inicio": ("2020-03-23", "2020-05-31"),
    "greve_caminhoneiros_2018": ("2018-05-21", "2018-06-01"),
    "apagao_2023": ("2023-08-15", "2023-08-15"),
}

# PASTAS
BASE = Path("etapa2")
RAW = BASE / "dados_brutos"
OUT = BASE / "saidas"
FIG = BASE / "figuras"
INMET_DIR = BASE / "dados_inmet"
for p in (RAW, OUT, FIG, INMET_DIR):
    p.mkdir(parents=True, exist_ok=True)

DATA_EXECUCAO = datetime.now()

# estilo dos gráficos
plt.rcParams.update({
    "figure.figsize": (11, 4.2), "figure.dpi": 110, "savefig.dpi": 200,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#DDDDDD", "grid.linewidth": 0.6,
    "axes.titlesize": 12, "axes.titleweight": "bold", "font.size": 10,
    "lines.linewidth": 1.0,
})
COR_SERIE = "#1F5F8B"
COR_REF = "#7A7A7A"
COR_ALERTA = "#C0392B"
COR_JANELA = "#E8F0F6"

print("Execução:", DATA_EXECUCAO.strftime("%Y-%m-%d %H:%M"))
print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)

## 2. Aquisição e inventário

Cada arquivo anual é baixado uma vez e guardado em `etapa2/dados_brutos/`.
Para rastreabilidade registramos a data e hora do download e o hash SHA-256 de cada arquivo.
Como o ONS revisa os dados após a publicação, o hash mostra qual versão foi usada.

In [ ]:
def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()

registros_download = []
for ano in range(ANO_INICIO_INVENTARIO, ANO_FIM + 1):
    destino = RAW / f"CARGA_ENERGIA_{ano}.csv"
    status = "cache local"
    if REBAIXAR or not destino.exists():
        try:
            urllib.request.urlretrieve(URL_ONS.format(ano=ano), destino)
            status = "baixado"
        except Exception as e:
            status = f"falhou: {e}"
    info = {"ano": ano, "arquivo": destino.name, "url": URL_ONS.format(ano=ano), "status": status}
    if destino.exists():
        info.update({
            "tamanho_bytes": destino.stat().st_size,
            "sha256": sha256(destino),
            "data_obtencao": datetime.fromtimestamp(destino.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
        })
    registros_download.append(info)

df_downloads = pd.DataFrame(registros_download)
df_downloads.to_csv(OUT / "registro_downloads.csv", index=False)
print(df_downloads["status"].value_counts().to_string())
falhas = df_downloads[~df_downloads["status"].isin(["baixado", "cache local"])]
if len(falhas):
    print("\nATENÇÃO — arquivos que não foram obtidos:")
    print(falhas[["ano", "status"]].to_string(index=False))
df_downloads[["ano", "status", "tamanho_bytes"]].tail()

### 2.1 Leitura e junção dos arquivos anuais

In [ ]:
COLUNAS_ESPERADAS = ["id_subsistema", "nom_subsistema", "din_instante", "val_cargaenergiamwmed"]

def ler_arquivo(caminho):
    for sep in (";", ",", "\t"):
        try:
            df = pd.read_csv(caminho, sep=sep, dtype=str, encoding="utf-8-sig")
        except UnicodeDecodeError:
            df = pd.read_csv(caminho, sep=sep, dtype=str, encoding="latin-1")
        if df.shape[1] >= 3:
            df.columns = [c.strip().lower() for c in df.columns]
            return df, sep
    raise ValueError(f"Não foi possível identificar o separador de {caminho.name}")

FORMATOS_DATA = ["%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M",
                 "%d/%m/%Y", "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M"]

def converter_datas(texto):
    texto = texto.astype("string").str.strip()
    datas = pd.Series(pd.NaT, index=texto.index, dtype="datetime64[ns]")
    formato = pd.Series(pd.NA, index=texto.index, dtype="string")
    for fmt in FORMATOS_DATA:
        faltam = datas.isna() & texto.notna()
        if not faltam.any():
            break
        conv = pd.to_datetime(texto[faltam], format=fmt, errors="coerce")
        ok = conv.notna()
        datas.loc[conv[ok].index] = conv[ok]
        formato.loc[conv[ok].index] = fmt
    return datas.dt.normalize(), formato

partes, esquemas = [], []
for _, linha in df_downloads.dropna(subset=["sha256"]).iterrows():
    caminho = RAW / linha["arquivo"]
    df_ano, sep = ler_arquivo(caminho)
    esquemas.append({"ano": linha["ano"], "separador": sep, "n_linhas": len(df_ano),
                     "colunas": "|".join(df_ano.columns)})
    df_ano["arquivo_origem"] = linha["arquivo"]
    df_ano["ano_arquivo"] = int(linha["ano"])
    partes.append(df_ano)

df_esquemas = pd.DataFrame(esquemas)
bruto = pd.concat(partes, ignore_index=True)

faltando = [c for c in COLUNAS_ESPERADAS if c not in bruto.columns]
if faltando:
    raise KeyError(f"Colunas esperadas ausentes: {faltando}. Colunas encontradas: {list(bruto.columns)}")

print(f"{len(partes)} arquivos lidos | {len(bruto):,} registros brutos | colunas: {COLUNAS_ESPERADAS}")
bruto.head()

### 2.2 Inventário

In [ ]:
datas_tmp, formato_data = converter_datas(bruto["din_instante"])
print("Formatos de data encontrados por arquivo:")
print(pd.crosstab(bruto["ano_arquivo"], formato_data.fillna("não reconhecido")).to_string())
inventario = pd.DataFrame([
    ("Fonte", f"Portal de Dados Abertos do ONS — {URL_PORTAL}"),
    ("Responsável / provedor", "Operador Nacional do Sistema Elétrico (ONS) — autor e mantenedor"),
    ("Formato", "CSV (UTF-8, separador '" + df_esquemas['separador'].mode()[0] + "'); também disponível em XLSX e Parquet"),
    ("Organização dos arquivos", f"1 arquivo por ano ({df_esquemas['ano'].min()}–{df_esquemas['ano'].max()}), {len(df_esquemas)} arquivos"),
    ("Período coberto", f"{datas_tmp.min():%d/%m/%Y} a {datas_tmp.max():%d/%m/%Y}"),
    ("Granularidade", "Diária, por subsistema do SIN"),
    ("Frequência de atualização", "Diária (12h e 19h), segundo o portal; dados sujeitos a revisão (consistência recorrente)"),
    ("Quantidade de registros (bruto)", f"{len(bruto):,}".replace(",", ".")),
    ("Registros por subsistema", "; ".join(f"{k}: {v:,}".replace(",", ".") for k, v in bruto['id_subsistema'].value_counts().sort_index().items())),
    ("Principais atributos", "id_subsistema, nom_subsistema, din_instante, val_cargaenergiamwmed"),
    ("Unidade da medida", "MWmed (megawatt médio)"),
    ("Licença / condições de uso", "Creative Commons Atribuição (CC-BY): permite reutilizar e adaptar, citando o ONS e informando as alterações; dados fornecidos 'como estão'"),
    ("Data da obtenção", DATA_EXECUCAO.strftime("%d/%m/%Y %H:%M")),
], columns=["Item", "Descrição"])

inventario.to_csv(OUT / "inventario.csv", index=False)
pd.set_option("display.max_colwidth", 140)
inventario

## 3. Conversão de tipos (sem alterar valores)

Convertemos para os tipos corretos guardando o texto original, para medir quantos valores não puderam ser convertidos.


In [ ]:
base = bruto.copy()
base["id_subsistema"] = base["id_subsistema"].str.strip().str.upper()
base["nom_subsistema"] = base["nom_subsistema"].str.strip()
base["data"] = datas_tmp
base["formato_data_origem"] = formato_data
base["carga_mwmed"] = pd.to_numeric(base["val_cargaenergiamwmed"].str.replace(",", ".", regex=False), errors="coerce")
base[["id_subsistema", "nom_subsistema", "data", "carga_mwmed"]].dtypes

## 4. Avaliação de qualidade (com evidências)

Cada verificação vai para a tabela `relatorio_qualidade.csv` com dimensão, regra, resultado e percentual.
Avaliamos a base inteira (todos os subsistemas e anos) e, à parte, o recorte do projeto (subsistema Sul).

In [ ]:
qualidade = []
def registrar(dimensao, verificacao, escopo, ocorrencias, total, observacao=""):
    pct = (100 * ocorrencias / total) if total else 0.0
    qualidade.append({"dimensao": dimensao, "verificacao": verificacao, "escopo": escopo,
                      "ocorrencias": int(ocorrencias), "total_avaliado": int(total),
                      "percentual": round(pct, 3), "observacao": observacao})
    print(f"[{dimensao:<12}] {verificacao:<62} {int(ocorrencias):>6} de {int(total):>7} ({pct:6.3f}%)  {observacao}")

br = lambda x: f"{x:+.2f}".replace(".", ",")

N = len(base)
SUBS_VALIDOS = {"N", "NE", "S", "SE"}

### 4.1 Completude

In [ ]:
for col in COLUNAS_ESPERADAS:
    n_nulos = bruto[col].isna().sum() + (bruto[col].astype(str).str.strip() == "").sum()
    registrar("Completude", f"Valores ausentes em '{col}'", "Base inteira", n_nulos, N)

lacunas_por_sub = []
dias_ausentes = {}
fim_global = base["data"].max()
for sub, g in base.dropna(subset=["data"]).groupby("id_subsistema"):
    esperado = pd.date_range(g["data"].min(), fim_global, freq="D")
    presentes = set(g["data"].unique())
    ausentes = [d for d in esperado if d not in presentes]
    dias_ausentes[sub] = pd.DatetimeIndex(ausentes)
    registrar("Completude", f"Dias sem registro no calendário — subsistema {sub}", f"Subsistema {sub}",
              len(ausentes), len(esperado), f"{g['data'].min():%d/%m/%Y}–{fim_global:%d/%m/%Y}")

aus_sul = pd.Series(1, index=dias_ausentes.get(SUBSISTEMA_ALVO, pd.DatetimeIndex([])))
tabela_aus_ano = aus_sul.groupby(aus_sul.index.year).sum().rename("dias_ausentes") if len(aus_sul) else pd.Series(dtype=int, name="dias_ausentes")
tabela_aus_ano

### 4.2 Validade

In [ ]:
texto_preenchido = bruto["val_cargaenergiamwmed"].notna() & (bruto["val_cargaenergiamwmed"].str.strip() != "")
registrar("Validade", "Carga não numérica (texto não convertível)", "Base inteira",
          (texto_preenchido & base["carga_mwmed"].isna()).sum(), texto_preenchido.sum())
registrar("Validade", "Carga menor ou igual a zero", "Base inteira",
          (base["carga_mwmed"] <= 0).sum(), base["carga_mwmed"].notna().sum(), "fisicamente impossível")

datas_preenchidas = bruto["din_instante"].notna()
registrar("Validade", "Data não reconhecível", "Base inteira",
          (datas_preenchidas & base["data"].isna()).sum(), datas_preenchidas.sum())
registrar("Validade", "Data no futuro (após a data de execução)", "Base inteira",
          (base["data"] > pd.Timestamp(DATA_EXECUCAO).normalize()).sum(), base["data"].notna().sum())
registrar("Validade", "Código de subsistema fora de {N, NE, S, SE}", "Base inteira",
          (~base["id_subsistema"].isin(SUBS_VALIDOS)).sum(), N,
          "códigos encontrados: " + ", ".join(sorted(base["id_subsistema"].dropna().unique())))

# faixa plausível por subsistema
faixas = base.groupby("id_subsistema")["carga_mwmed"].describe(percentiles=[.001, .5, .999])[["min", "0.1%", "50%", "99.9%", "max"]].round(0)
faixas

### 4.3 Unicidade

In [ ]:
chave = ["id_subsistema", "data"]
dup_exatas = base.duplicated(subset=chave + ["carga_mwmed"], keep="first").sum()
dup_chave = base.duplicated(subset=chave, keep=False)
conflitos = base[dup_chave].groupby(chave)["carga_mwmed"].nunique()
n_conflitos = int((conflitos > 1).sum())

registrar("Unicidade", "Registros duplicados idênticos (subsistema + data + valor)", "Base inteira", dup_exatas, N)
registrar("Unicidade", "Chaves (subsistema + data) com valores diferentes", "Base inteira",
          n_conflitos, base[chave].drop_duplicates().shape[0], "duplicata conflitante")

grp = base[dup_chave].groupby(chave)["carga_mwmed"]
parciais = (grp.apply(lambda x: x.isna().any() and x.notna().any()))
chaves_parciais = parciais[parciais].index.tolist()
registrar("Unicidade", "Chaves repetidas em que uma linha tem valor e outra está vazia", "Base inteira",
          len(chaves_parciais), base[chave].drop_duplicates().shape[0],
          "; ".join(f"{s_} em {d_:%d/%m/%Y}" for s_, d_ in chaves_parciais[:10]))

### 4.4 Consistência

In [ ]:
# (a) Esquema igual em todos os arquivos anuais
esquema_ref = df_esquemas["colunas"].mode()[0]
anos_diferentes = df_esquemas[df_esquemas["colunas"] != esquema_ref]
registrar("Consistência", "Arquivos anuais com colunas diferentes do padrão", "Arquivos",
          len(anos_diferentes), len(df_esquemas),
          "" if anos_diferentes.empty else "anos: " + ", ".join(map(str, anos_diferentes["ano"])))

# (a2) Formato de data igual em todos os arquivos
fmt_por_ano = base.dropna(subset=["formato_data_origem"]).groupby("ano_arquivo")["formato_data_origem"].agg(lambda x: x.mode()[0])
fmt_padrao = fmt_por_ano.mode()[0]
anos_fmt_dif = fmt_por_ano[fmt_por_ano != fmt_padrao]
registrar("Consistência", "Arquivos anuais com formato de data diferente do padrão", "Arquivos",
          len(anos_fmt_dif), len(fmt_por_ano),
          f"padrão '{fmt_padrao}'" + ("" if anos_fmt_dif.empty else "; diferente em: " +
          ", ".join(f"{a} ('{f}')" for a, f in anos_fmt_dif.items())))

# (b) Cada código de subsistema corresponde a um único nome
nomes_por_id = base.groupby("id_subsistema")["nom_subsistema"].nunique()
registrar("Consistência", "Códigos de subsistema com mais de um nome", "Base inteira",
          (nomes_por_id > 1).sum(), len(nomes_por_id),
          "; ".join(f"{k}={'/'.join(sorted(base.loc[base.id_subsistema==k,'nom_subsistema'].dropna().unique()))}"
                    for k in nomes_por_id.index))

# (c) Datas dentro do ano do arquivo
fora_do_ano = (base["data"].dt.year != base["ano_arquivo"]) & base["data"].notna()
registrar("Consistência", "Registros com data fora do ano do arquivo de origem", "Base inteira", fora_do_ano.sum(), N)

# (d) quebra de série nos marcos do ONS, 28 dias antes x 28 depois
# comparado com as mesmas datas em 3 anos de controle
sul = base[(base["id_subsistema"] == SUBSISTEMA_ALVO)].dropna(subset=["data"]).groupby("data")["carga_mwmed"].mean()

def variacao_janela(m, dias=28):
    antes = sul[(sul.index >= m - pd.Timedelta(days=dias)) & (sul.index < m)].mean()
    depois = sul[(sul.index >= m) & (sul.index < m + pd.Timedelta(days=dias))].mean()
    return 100 * (depois / antes - 1)

def degrau(marco, n_controles=3):
    anos_controle, k = [], 1
    while len(anos_controle) < n_controles and k < 10:
        m = marco - pd.DateOffset(years=k)
        if m.year != 2020 and (m - pd.Timedelta(days=28)) >= sul.index.min():
            anos_controle.append(m)
        k += 1
    v_marco = variacao_janela(marco)
    v_controle = float(np.mean([variacao_janela(m) for m in anos_controle]))
    return v_marco, v_controle, v_marco - v_controle, [m.year for m in anos_controle]

degraus = []
for nome, marco in [("Inclusão de usinas não despachadas (mar/2021)", MARCO_METODOLOGIA_1),
                    ("Inclusão de MMGD (abr/2023)", MARCO_METODOLOGIA_2)]:
    v, c, d, anos_c = degrau(marco)
    degraus.append({"marco": nome, "data": marco.date(), "variacao_no_marco_%": round(v, 2),
                    "variacao_media_anos_controle_%": round(c, 2), "anos_controle": anos_c,
                    "efeito_estimado_pp": round(d, 2)})
df_degraus = pd.DataFrame(degraus)
for _, r in df_degraus.iterrows():
    qualidade.append({"dimensao": "Consistência", "verificacao": f"Degrau na série — {r['marco']}",
                      "escopo": f"Subsistema {SUBSISTEMA_ALVO}", "ocorrencias": np.nan, "total_avaliado": np.nan,
                      "percentual": r["efeito_estimado_pp"],
                      "observacao": (f"variação de {br(r['variacao_no_marco_%'])}% entre as 4 semanas antes e depois do marco, "
                                     f"contra {br(r['variacao_media_anos_controle_%'])}% nas mesmas datas de "
                                     f"{', '.join(map(str, r['anos_controle']))}: efeito estimado de "
                                     f"{br(r['efeito_estimado_pp'])} p.p.")})
df_degraus

### 4.5 Atualidade

In [ ]:
ultima = base.groupby("id_subsistema")["data"].max()
hoje = pd.Timestamp(DATA_EXECUCAO).normalize()
for sub, d in ultima.items():
    atraso = (hoje - d).days
    qualidade.append({"dimensao": "Atualidade", "verificacao": f"Defasagem do último dado — subsistema {sub}",
                      "escopo": f"Subsistema {sub}", "ocorrencias": atraso, "total_avaliado": np.nan,
                      "percentual": np.nan, "observacao": f"último dia: {d:%d/%m/%Y}; defasagem de {atraso} dia(s)"})
    print(f"[Atualidade  ] Subsistema {sub}: último dia {d:%d/%m/%Y} — defasagem de {atraso} dia(s)")

### 4.6 Acurácia

Usamos duas evidências indiretas.
1. Outliers estatísticos. Cada dia é comparado com a mediana do mesmo dia da semana nas 3 semanas anteriores e 3 posteriores,
   assim o domingo, naturalmente mais baixo, não é confundido com anomalia. A distância é medida em z-score robusto (MAD).
2. Eventos conhecidos. Se os dados forem fiéis à realidade, quedas reais (início da pandemia, apagão de 15/08/2023) precisam aparecer.

In [ ]:
def z_robusto(serie, semanas=3):
    esperado = pd.Series(np.nan, index=serie.index)
    for dow in range(7):
        s = serie[serie.index.dayofweek == dow]
        esperado.loc[s.index] = s.rolling(2 * semanas + 1, center=True, min_periods=semanas).median()
    residuo = serie / esperado - 1
    centro = residuo.median()
    mad = (residuo - centro).abs().median()
    return 0.6745 * (residuo - centro) / mad, esperado

z_sul, mediana_sul = z_robusto(sul)
outliers_sul = sul[z_sul.abs() > LIMIAR_Z_ROBUSTO]
registrar("Acurácia", f"Dias atípicos (|z robusto| > {LIMIAR_Z_ROBUSTO}) — mantidos e sinalizados", f"Subsistema {SUBSISTEMA_ALVO}",
          len(outliers_sul), sul.notna().sum())

# Eventos conhecidos
ini, fim = map(pd.Timestamp, EVENTOS_CONHECIDOS["pandemia_inicio"])
media_2020 = sul[(sul.index >= ini) & (sul.index <= fim)].mean()
media_2019 = sul[(sul.index >= ini - pd.DateOffset(years=1)) & (sul.index <= fim - pd.DateOffset(years=1))].mean()
var_pandemia = 100 * (media_2020 / media_2019 - 1)
qualidade.append({"dimensao": "Acurácia", "verificacao": "Evento conhecido: início da pandemia (23/03–31/05/2020 vs 2019)",
                  "escopo": f"Subsistema {SUBSISTEMA_ALVO}", "ocorrencias": np.nan, "total_avaliado": np.nan,
                  "percentual": round(var_pandemia, 2),
                  "observacao": f"carga média {br(var_pandemia)}% em relação às mesmas datas de 2019 (queda esperada)"})
print(f"[Acurácia    ] Pandemia: carga média {var_pandemia:+.2f}% em relação ao mesmo período de 2019")

dia_apagao = pd.Timestamp(EVENTOS_CONHECIDOS["apagao_2023"][0])
if dia_apagao in sul.index:
    mesmos_dias = [dia_apagao - pd.Timedelta(weeks=k) for k in range(1, 5)]
    ref = sul.reindex(mesmos_dias).median()
    var_apagao = 100 * (sul[dia_apagao] / ref - 1)
    qualidade.append({"dimensao": "Acurácia", "verificacao": "Evento conhecido: apagão de 15/08/2023 (vs mesmo dia da semana nas 4 semanas anteriores)",
                      "escopo": f"Subsistema {SUBSISTEMA_ALVO}", "ocorrencias": np.nan, "total_avaliado": np.nan,
                      "percentual": round(var_apagao, 2),
                      "observacao": f"carga do dia {br(var_apagao)}% em relação à mediana do mesmo dia da semana nas 4 semanas anteriores"})
    print(f"[Acurácia    ] Apagão 15/08/2023: carga {var_apagao:+.2f}% em relação ao mesmo dia da semana nas 4 semanas anteriores")

# dias atípicos com explicação conhecida
def _feriados_ano(ano):
    from datetime import date as _d, timedelta as _td
    a = ano % 19; b, c = divmod(ano, 100); d, e = divmod(b, 4); f = (b + 8) // 25
    g = (b - f + 1) // 3; h = (19 * a + b - d - g + 15) % 30; i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7; m = (a + 11 * h + 22 * l) // 451
    mes, dia = divmod(h + l - 7 * m + 114, 31); p = _d(ano, mes, dia + 1)
    fixos = [(1, 1), (4, 21), (5, 1), (9, 7), (10, 12), (11, 2), (11, 15), (12, 25)] + ([(11, 20)] if ano >= 2024 else [])
    return {_d(ano, mm, dd) for mm, dd in fixos} | {p - _td(days=48), p - _td(days=47), p - _td(days=2), p + _td(days=60)}

feriados_set = set().union(*[_feriados_ano(a) for a in range(sul.index.min().year, sul.index.max().year + 1)])
md_ = outliers_sul.index.strftime("%m-%d")
em_feriado = pd.Series([d.date() in feriados_set for d in outliers_sul.index], index=outliers_sul.index)
fim_de_ano = pd.Series((md_ >= "12-20") | (md_ <= "01-07"), index=outliers_sul.index)
em_evento = pd.Series(False, index=outliers_sul.index)
for ini_, fim_ in EVENTOS_CONHECIDOS.values():
    em_evento |= (outliers_sul.index >= ini_) & (outliers_sul.index <= fim_)
causa = np.select([em_feriado, fim_de_ano & ~em_feriado, em_evento & ~em_feriado & ~fim_de_ano],
                  ["feriado nacional / Carnaval", "período de Natal e Ano-Novo (20/12–07/01)", "evento conhecido"],
                  default="outros (ex.: ondas de calor/frio, pontes de feriado)")
composicao = pd.Series(causa).value_counts().rename("dias").to_frame()
composicao["%"] = (100 * composicao["dias"] / composicao["dias"].sum()).round(1)
explicados = int(composicao.drop(index="outros (ex.: ondas de calor/frio, pontes de feriado)", errors="ignore")["dias"].sum())
qualidade.append({"dimensao": "Acurácia", "verificacao": "Dias atípicos com causa conhecida (feriado, fim de ano ou evento)",
                  "escopo": f"Subsistema {SUBSISTEMA_ALVO}", "ocorrencias": explicados, "total_avaliado": len(outliers_sul),
                  "percentual": round(100 * explicados / max(1, len(outliers_sul)), 3),
                  "observacao": "; ".join(f"{k}: {v}" for k, v in composicao["dias"].items())})
composicao

### 4.7 Relatório de qualidade consolidado

In [ ]:
relatorio_qualidade = pd.DataFrame(qualidade)
relatorio_qualidade.to_csv(OUT / "relatorio_qualidade.csv", index=False)
relatorio_qualidade

### 4.8 Figuras de apoio

In [ ]:
janela_ini = pd.Timestamp(JANELA_INICIO)

# Figura 1, série completa com marcos e janela
fig, ax = plt.subplots()
ax.axvspan(janela_ini, sul.index.max(), color=COR_JANELA, zorder=0, label="Janela usada no projeto")
ax.plot(sul.index, sul.values, color=COR_SERIE, linewidth=0.6)
import matplotlib.transforms as mtransforms
topo = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
for i, (marco, rotulo) in enumerate([(MARCO_METODOLOGIA_1, "mar/2021"), (MARCO_METODOLOGIA_2, "abr/2023")]):
    ax.axvline(marco, color=COR_REF, linestyle="--", linewidth=1,
               label="Mudança de metodologia do ONS" if i == 0 else None)
    ax.text(marco, 1.01, rotulo, transform=topo, ha="right" if i == 0 else "left",
            va="bottom", fontsize=8, color=COR_REF)
ax.set_title("Carga diária — subsistema Sul (MWmed)", pad=18)
ax.set_ylabel("MWmed")
ax.legend(loc="upper left", frameon=False)
fig.tight_layout(); fig.savefig(FIG / "fig1_serie_sul_metodologia.png"); plt.show()

# Figura 2, dias ausentes por ano (Sul)
anos = pd.Index(range(sul.index.min().year, sul.index.max().year + 1), name="ano")
aus = tabela_aus_ano.reindex(anos, fill_value=0)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(aus.index, aus.values, color=COR_SERIE, width=0.7)
for x, y in zip(aus.index, aus.values):
    if y > 0:
        ax.text(x, y, str(int(y)), ha="center", va="bottom", fontsize=8)
ax.set_title("Dias sem registro por ano — subsistema Sul")
ax.set_ylabel("dias"); ax.set_xticks(aus.index); ax.tick_params(axis="x", rotation=90)
ax.set_ylim(0, max(1, aus.max()) * 1.25); ax.grid(axis="x", visible=False)
fig.tight_layout(); fig.savefig(FIG / "fig2_dias_ausentes_por_ano.png"); plt.show()

# Figura 3, dias atípicos sinalizados
fig, ax = plt.subplots()
ax.plot(sul.index, sul.values, color=COR_SERIE, linewidth=0.5, label="Carga diária")
ax.scatter(outliers_sul.index, outliers_sul.values, s=14, color=COR_ALERTA, zorder=3,
           label=f"Dia atípico (|z| > {LIMIAR_Z_ROBUSTO})")
ax.set_title("Dias atípicos detectados — subsistema Sul")
ax.set_ylabel("MWmed"); ax.legend(loc="lower right", frameon=False)
fig.tight_layout(); fig.savefig(FIG / "fig3_dias_atipicos.png"); plt.show()

## 5. Tratamento dos dados


In [ ]:
log = []
def registrar_transformacao(dado_original, problema, transformacao, justificativa, afetados, linhas_resultado):
    log.append({"passo": len(log) + 1, "dado_original": dado_original, "problema_identificado": problema,
                "transformacao": transformacao, "justificativa": justificativa,
                "registros_afetados": int(afetados), "linhas_apos_passo": int(linhas_resultado)})
    print(f"T{len(log):02d} | {transformacao:<70} afetados: {int(afetados):>6} | linhas: {int(linhas_resultado):,}")

# T1, junção dos arquivos anuais
registrar_transformacao(f"{len(df_esquemas)} arquivos CSV anuais", "Base fragmentada por ano",
    "Concatenação em uma base única com coluna 'arquivo_origem'",
    "Permite análise da série contínua sem perder a origem de cada registro", len(bruto), len(bruto))

# T2, tipos
df = base.copy()
n_tipo = (df["data"].isna() | df["carga_mwmed"].isna()).sum()
registrar_transformacao("din_instante e val_cargaenergiamwmed como texto", "Tipos inadequados para análise temporal",
    "Conversão para data (din_instante → data, aceitando os formatos com e sem hora usados pelo ONS) e número decimal (→ carga_mwmed)",
    "Campos precisam de tipo correto para ordenação, cálculo e modelagem; valores não convertíveis viram nulos e são tratados adiante",
    n_tipo, len(df))

# T3, registros sem data (não recuperáveis)
sem_data = df["data"].isna().sum()
df = df.dropna(subset=["data"])
registrar_transformacao("Registros sem data válida", "Impossível posicionar o registro na série",
    "Remoção dos registros sem data", "Um valor de carga sem data não tem uso em série temporal", sem_data, len(df))

# T4, duplicatas
antes = len(df)
df["_tem_valor"] = df["carga_mwmed"].notna()
df = (df.sort_values(["id_subsistema", "data", "_tem_valor", "arquivo_origem"], kind="stable")
        .drop_duplicates(subset=["id_subsistema", "data"], keep="last")
        .drop(columns="_tem_valor"))
registrar_transformacao("Registros repetidos para subsistema + data", "Violação de unicidade",
    "Manter um único registro por subsistema e data, priorizando a linha com valor preenchido (e, entre elas, a do arquivo mais recente)",
    "Cada dia deve ter uma única medida; quando uma das cópias está vazia, a cópia com valor preserva a informação real em vez de forçar uma imputação",
    antes - len(df), len(df))

# T5, filtro do subsistema
antes = len(df)
df = df[df["id_subsistema"] == SUBSISTEMA_ALVO].copy()
registrar_transformacao("Base com 4 subsistemas (N, NE, S, SE)", "Escopo do projeto é a área de atuação da distribuidora (subsistema Sul)",
    f"Filtro id_subsistema = '{SUBSISTEMA_ALVO}' (Sul)", "Alinha os dados à organização definida na Etapa 1",
    antes - len(df), len(df))

# T6, recorte temporal
antes = len(df)
df = df[df["data"] >= janela_ini].copy()
registrar_transformacao(f"Série desde {sul.index.min():%Y}", "Série longa atravessa mudanças de metodologia e padrões de consumo antigos",
    f"Recorte a partir de {janela_ini:%d/%m/%Y}",
    "Equilíbrio entre volume de dados (vários ciclos anuais) e representatividade do consumo atual; as quebras restantes são tratadas com a variável regime_metodologico",
    antes - len(df), len(df))

# T7, valores inválidos
invalidos = (df["carga_mwmed"] <= 0).sum()
df.loc[df["carga_mwmed"] <= 0, "carga_mwmed"] = np.nan
registrar_transformacao("Carga ≤ 0", "Valor fisicamente impossível (falha de medição/registro)",
    "Substituição por nulo para posterior tratamento", "Evita que valores impossíveis distorçam o modelo", invalidos, len(df))

# T8, calendário completo
df = df.set_index("data").sort_index()
calendario = pd.date_range(df.index.min(), df.index.max(), freq="D", name="data")
dias_novos = len(calendario) - len(df)
df = df.reindex(calendario)
df["flag_ausente_original"] = df["carga_mwmed"].isna().astype(int)
registrar_transformacao("Dias sem registro no calendário", "Série com buracos de datas",
    "Reindexação para calendário diário completo (dias faltantes entram como nulos); flag_ausente_original = 1 para todo dia sem valor válido na origem",
    "Modelos de série temporal exigem espaçamento regular; a flag preserva a informação de que o dado original não existia ou era inválido",
    dias_novos, len(df))

# T9, imputação de lacunas curtas
nulos = df["carga_mwmed"].isna()
grupo = (nulos != nulos.shift()).cumsum()
tam_lacuna = nulos.groupby(grupo).transform("sum")
curtas = nulos & (tam_lacuna <= MAX_LACUNA_IMPUTAVEL)
interp = df["carga_mwmed"].interpolate(method="time", limit_area="inside")
df.loc[curtas, "carga_mwmed"] = interp[curtas]
df["flag_imputado"] = (curtas & df["carga_mwmed"].notna()).astype(int)
restantes = df["carga_mwmed"].isna().sum()
registrar_transformacao(f"Lacunas de até {MAX_LACUNA_IMPUTAVEL} dias consecutivos", "Valores ausentes",
    "Interpolação linear no tempo, com flag_imputado = 1",
    f"Lacunas curtas são bem aproximadas pela vizinhança; lacunas maiores ({restantes} dia(s)) ficam nulas para não inventar padrão",
    int(df["flag_imputado"].sum()), len(df))

# T10, dias atípicos (sinalizar, não remover)
z, _ = z_robusto(df["carga_mwmed"])
df["flag_atipico"] = (z.abs() > LIMIAR_Z_ROBUSTO).astype(int)
registrar_transformacao("Dias com carga muito distante da vizinhança", "Possíveis outliers",
    f"Sinalização com flag_atipico (|z robusto| > {LIMIAR_Z_ROBUSTO}); valores mantidos",
    "A maioria corresponde a eventos reais (feriados, pandemia, apagão); removê-los esconderia comportamento que o modelo precisa conhecer",
    int(df["flag_atipico"].sum()), len(df))

### 5.1 Variáveis derivadas

Feriados nacionais são calculados no próprio notebook (datas fixas + datas móveis a partir da Páscoa), sem depender de pacotes externos.
Carnaval é incluído porque, embora seja ponto facultativo, reduz fortemente o consumo.

In [ ]:
def pascoa(ano):
    a = ano % 19; b, c = divmod(ano, 100); d, e = divmod(b, 4); f = (b + 8) // 25
    g = (b - f + 1) // 3; h = (19 * a + b - d - g + 15) % 30; i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7; m = (a + 11 * h + 22 * l) // 451
    mes, dia = divmod(h + l - 7 * m + 114, 31)
    return date(ano, mes, dia + 1)

def feriados_nacionais(ano):
    p = pascoa(ano)
    fer = {
        date(ano, 1, 1): "Confraternização Universal", date(ano, 4, 21): "Tiradentes",
        date(ano, 5, 1): "Dia do Trabalho", date(ano, 9, 7): "Independência",
        date(ano, 10, 12): "Nossa Senhora Aparecida", date(ano, 11, 2): "Finados",
        date(ano, 11, 15): "Proclamação da República", date(ano, 12, 25): "Natal",
        p - timedelta(days=48): "Carnaval (segunda)", p - timedelta(days=47): "Carnaval (terça)",
        p - timedelta(days=2): "Sexta-feira Santa", p + timedelta(days=60): "Corpus Christi",
    }
    if ano >= 2024:
        fer[date(ano, 11, 20)] = "Consciência Negra"
    return fer

todos_feriados = {}
for ano in range(df.index.min().year, df.index.max().year + 1):
    todos_feriados.update(feriados_nacionais(ano))

idx = df.index
df["dia_semana"] = idx.dayofweek
df["nome_dia_semana"] = idx.dayofweek.map(dict(enumerate(["seg", "ter", "qua", "qui", "sex", "sab", "dom"])))
df["fim_de_semana"] = (idx.dayofweek >= 5).astype(int)
df["mes"] = idx.month
df["ano"] = idx.year

def estacao(d):
    md_ = (d.month, d.day)
    if md_ >= (12, 21) or md_ < (3, 20): return "verao"
    if md_ < (6, 21): return "outono"
    if md_ < (9, 22): return "inverno"
    return "primavera"
df["estacao"] = [estacao(d) for d in idx]

df["feriado"] = [int(d.date() in todos_feriados) for d in idx]
df["nome_feriado"] = [todos_feriados.get(d.date(), "") for d in idx]

df["regime_metodologico"] = np.select(
    [idx < MARCO_METODOLOGIA_1, idx < MARCO_METODOLOGIA_2], [1, 2], default=3)

df["flag_evento_conhecido"] = 0
for ini_, fim_ in EVENTOS_CONHECIDOS.values():
    df.loc[(idx >= ini_) & (idx <= fim_), "flag_evento_conhecido"] = 1

registrar_transformacao("Somente data e carga", "Ausência de variáveis explicativas de calendário",
    "Criação de dia_semana, fim_de_semana, mes, ano, estacao, feriado, nome_feriado",
    "Consumo de energia varia por dia da semana, estação e feriados; variáveis previstas na Etapa 1", 0, len(df))
registrar_transformacao("Série com 3 metodologias de apuração", "Quebra de série documentada pelo ONS",
    "Criação de regime_metodologico (1: até fev/2021; 2: mar/2021–abr/2023; 3: a partir de 29/04/2023)",
    "Permite ao modelo distinguir mudança de metodologia de mudança real de consumo", 0, len(df))
registrar_transformacao("Períodos de comportamento excepcional", "Eventos raros podem enviesar o modelo",
    "Criação de flag_evento_conhecido (greve dos caminhoneiros 2018, início da pandemia, apagão de 15/08/2023)",
    "Permite excluir ou ponderar esses períodos no treino sem apagar os dados", int(df["flag_evento_conhecido"].sum()), len(df))

df[["carga_mwmed", "dia_semana", "estacao", "feriado", "regime_metodologico", "flag_imputado", "flag_atipico"]].head()

### 5.2 Temperatura média diária do INMET (opcional)

Para ativar, use `USAR_INMET = True` na seção 1. Antes, baixe os dados históricos em <https://portal.inmet.gov.br/dadoshistoricos>
(um ZIP por ano), extraia os CSVs das estações de Curitiba (A807), Florianópolis (A806) e Porto Alegre (A801)
e coloque em `etapa2/dados_inmet/`.

In [ ]:
if USAR_INMET:
    ESTACOES = ("A807", "A806", "A801")
    arquivos = [a for a in sorted(INMET_DIR.glob("*.CSV")) + sorted(INMET_DIR.glob("*.csv"))
                if any(e in a.name.upper() for e in ESTACOES)]
    if not arquivos:
        raise FileNotFoundError(f"Nenhum CSV das estações {ESTACOES} em {INMET_DIR}")
    diarios, horas_total, horas_faltando = [], 0, 0
    for arq in arquivos:
        t = pd.read_csv(arq, sep=";", skiprows=8, encoding="latin-1", decimal=",", dtype=str)
        t.columns = [c.strip().upper() for c in t.columns]
        col_data = next(c for c in t.columns if c.startswith("DATA"))
        col_temp = next(c for c in t.columns if "BULBO SECO" in c)
        temp = pd.to_numeric(t[col_temp].str.replace(",", ".", regex=False), errors="coerce")
        temp = temp.mask(temp <= -9999)                              # INMET usa -9999 para ausente
        datas = pd.to_datetime(t[col_data].str.replace("/", "-", regex=False), errors="coerce")
        horas_total += len(temp); horas_faltando += temp.isna().sum()
        estacao_id = next(e for e in ESTACOES if e in arq.name.upper())
        diarios.append(pd.DataFrame({"data": datas, "estacao": estacao_id, "temp": temp})
                       .dropna(subset=["data"]).groupby(["data", "estacao"])["temp"].mean().reset_index())
    temp_diaria = (pd.concat(diarios).groupby(["data", "estacao"])["temp"].mean()
                   .groupby("data").mean().rename("temperatura_media_c"))
    registrar("Completude", "Leituras horárias de temperatura ausentes (INMET, 3 estações)", "INMET",
              horas_faltando, horas_total)
    df = df.join(temp_diaria, how="left")
    sem_temp = df["temperatura_media_c"].isna().sum()
    registrar_transformacao("CSVs horários do INMET (3 estações)", "Granularidade horária e por estação; -9999 como ausente",
        "-9999 → nulo; média diária por estação; média das 3 capitais; junção pela data",
        f"Temperatura explica parte da variação do consumo (variável prevista na Etapa 1); {sem_temp} dia(s) ficaram sem temperatura",
        sem_temp, len(df))
else:
    print("Seção INMET desativada (USAR_INMET = False).")

## 6. Dataset tratado, dicionário de dados e log de transformações

In [ ]:
# atualiza o relatório de qualidade
relatorio_qualidade = pd.DataFrame(qualidade)
relatorio_qualidade.to_csv(OUT / "relatorio_qualidade.csv", index=False)

colunas_finais = ["id_subsistema", "nom_subsistema", "carga_mwmed", "dia_semana", "nome_dia_semana",
                  "fim_de_semana", "mes", "ano", "estacao", "feriado", "nome_feriado",
                  "regime_metodologico", "flag_ausente_original", "flag_imputado", "flag_atipico",
                  "flag_evento_conhecido", "arquivo_origem"]
if "temperatura_media_c" in df.columns:
    colunas_finais.insert(3, "temperatura_media_c")

df["id_subsistema"] = SUBSISTEMA_ALVO
df["nom_subsistema"] = df["nom_subsistema"].ffill().bfill()
df["arquivo_origem"] = df["arquivo_origem"].fillna("(dia criado na reindexação)")
dataset = df[colunas_finais].reset_index().rename(columns={"index": "data"})
dataset["data"] = dataset["data"].dt.strftime("%Y-%m-%d")
dataset.to_csv(OUT / "dataset_tratado_sul.csv", index=False, encoding="utf-8")

df_log = pd.DataFrame(log)
df_log.to_csv(OUT / "log_transformacoes.csv", index=False)

significados = {
    "data": ("Dia de referência da medição", "Date"),
    "id_subsistema": ("Código do subsistema do SIN (origem: ONS)", "String"),
    "nom_subsistema": ("Nome do subsistema (origem: ONS)", "String"),
    "carga_mwmed": ("Carga média de energia no dia, em MWmed (variável alvo; origem: val_cargaenergiamwmed)", "Float"),
    "temperatura_media_c": ("Temperatura média diária das capitais do Sul, em °C (INMET)", "Float"),
    "dia_semana": ("Dia da semana (0 = segunda, 6 = domingo)", "Integer"),
    "nome_dia_semana": ("Dia da semana abreviado", "String"),
    "fim_de_semana": ("1 se sábado ou domingo", "Integer (0/1)"),
    "mes": ("Mês do ano", "Integer"),
    "ano": ("Ano", "Integer"),
    "estacao": ("Estação do ano (hemisfério sul)", "String"),
    "feriado": ("1 se feriado nacional ou Carnaval", "Integer (0/1)"),
    "nome_feriado": ("Nome do feriado, vazio se dia comum", "String"),
    "regime_metodologico": ("Metodologia de apuração do ONS vigente no dia (1, 2 ou 3)", "Integer"),
    "flag_ausente_original": ("1 se o dia não tinha valor válido na origem (ausente, nulo, não numérico ou ≤ 0)", "Integer (0/1)"),
    "flag_imputado": ("1 se a carga foi estimada por interpolação", "Integer (0/1)"),
    "flag_atipico": ("1 se o dia foi detectado como atípico (outlier)", "Integer (0/1)"),
    "flag_evento_conhecido": ("1 se o dia está em um evento excepcional conhecido", "Integer (0/1)"),
    "arquivo_origem": ("Arquivo anual do ONS de onde veio o registro (rastreabilidade)", "String"),
}

def dominio(serie):
    s = serie.dropna()
    if s.empty:
        return "—"
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 12:
        return f"{s.min():,.0f} a {s.max():,.0f}".replace(",", ".")
    unicos = s.unique()
    try:
        vals = [str(v) for v in sorted(unicos, key=float)]
    except (TypeError, ValueError):
        vals = sorted(str(v) for v in unicos)
    if pd.api.types.is_numeric_dtype(s) and len(vals) > 3:
        return f"{vals[0]} a {vals[-1]}"
    return ", ".join(vals) if len(vals) <= 12 else f"{len(vals)} valores distintos"

linhas_dic = []
for col in dataset.columns:
    sig, tipo = significados.get(col, ("", str(dataset[col].dtype)))
    serie = dataset[col].replace("", np.nan) if col == "nome_feriado" else dataset[col]
    pct_nulo = 100 * serie.isna().mean()
    if col == "nome_feriado":
        qual = "Preenchido apenas em feriados"
    elif col == "carga_mwmed":
        qual = (f"{pct_nulo:.2f}% ausentes; {dataset['flag_imputado'].mean()*100:.2f}% imputados; "
                f"{dataset['flag_atipico'].mean()*100:.2f}% sinalizados como atípicos")
    elif col == "data":
        qual = f"Completo; calendário diário contínuo ({dataset['data'].iloc[0]} a {dataset['data'].iloc[-1]})"
    else:
        qual = "Completo" if pct_nulo == 0 else f"{pct_nulo:.2f}% ausentes"
    dominio_col = (f"{dataset['data'].iloc[0]} a {dataset['data'].iloc[-1]}" if col == "data" else dominio(serie))
    linhas_dic.append({"Campo": col, "Significado": sig, "Tipo": tipo, "Domínio": dominio_col, "Qualidade": qual})

dicionario = pd.DataFrame(linhas_dic)
dicionario.to_csv(OUT / "dicionario_dados.csv", index=False)

print(f"Dataset tratado: {len(dataset):,} linhas x {dataset.shape[1]} colunas → {OUT/'dataset_tratado_sul.csv'}")
dicionario

In [ ]:
df_log

### 6.1 Figuras do dataset tratado

In [ ]:
# Figura 4, perfil semanal (sazonalidade semanal)
fig, ax = plt.subplots(figsize=(8, 3.8))
ordem = ["seg", "ter", "qua", "qui", "sex", "sab", "dom"]
dados_box = [dataset.loc[dataset["nome_dia_semana"] == d, "carga_mwmed"].dropna() for d in ordem]
estilo_box = dict(showfliers=False, patch_artist=True,
                  boxprops=dict(facecolor="#D6E4EF", edgecolor=COR_SERIE), medianprops=dict(color=COR_SERIE, linewidth=1.5),
                  whiskerprops=dict(color=COR_REF), capprops=dict(color=COR_REF))
try:
    ax.boxplot(dados_box, tick_labels=ordem, **estilo_box)   # matplotlib >= 3.9
except TypeError:
    ax.boxplot(dados_box, labels=ordem, **estilo_box)        # versões anteriores
ax.set_title("Carga por dia da semana — subsistema Sul (dataset tratado)")
ax.set_ylabel("MWmed")
fig.tight_layout(); fig.savefig(FIG / "fig4_carga_por_dia_semana.png"); plt.show()

# Figura 5, perfil mensal médio (sazonalidade anual)
mensal = dataset.groupby("mes")["carga_mwmed"].mean()
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(mensal.index, mensal.values, color=COR_SERIE, linewidth=2, marker="o", markersize=5)
ax.set_xticks(range(1, 13), ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"])
ax.set_title("Carga média por mês — subsistema Sul (dataset tratado)")
ax.set_ylabel("MWmed")
fig.tight_layout(); fig.savefig(FIG / "fig5_carga_media_por_mes.png"); plt.show()

## 7. Resumo dos resultados (para o relatório)

In [ ]:
def q(dimensao, contem):
    r = relatorio_qualidade[(relatorio_qualidade["dimensao"] == dimensao) &
                            (relatorio_qualidade["verificacao"].str.contains(contem, regex=False))]
    return r.iloc[0] if len(r) else None

fmt = lambda n: f"{int(n):,}".replace(",", ".")
pctf = lambda x: f"{x:.2f}".replace(".", ",")

linhas = []
linhas.append(f"# Resumo dos resultados — Etapa 2\n\nExecução: {DATA_EXECUCAO:%d/%m/%Y %H:%M}\n")
linhas.append("## Inventário")
linhas.append(f"- Base bruta: **{fmt(len(bruto))} registros** em {len(df_esquemas)} arquivos anuais "
              f"({datas_tmp.min():%d/%m/%Y} a {datas_tmp.max():%d/%m/%Y}), 4 subsistemas.")
linhas.append(f"- Subsistema Sul: {fmt((bruto['id_subsistema'].str.strip().str.upper()==SUBSISTEMA_ALVO).sum())} registros brutos.")

linhas.append("\n## Qualidade (evidências)")
for _, r in relatorio_qualidade.iterrows():
    if pd.notna(r["total_avaliado"]):
        linhas.append(f"- **{r['dimensao']}** — {r['verificacao']}: {fmt(r['ocorrencias'])} de {fmt(r['total_avaliado'])} "
                      f"({pctf(r['percentual'])}%). {r['observacao']}".rstrip())
    else:
        linhas.append(f"- **{r['dimensao']}** — {r['verificacao']}: {str(r['observacao']).rstrip('.')}.")

linhas.append("\n## Tratamento")
linhas.append(f"- {len(df_log)} transformações registradas (ver `log_transformacoes.csv`).")
linhas.append(f"- Dataset final: **{fmt(len(dataset))} dias** ({dataset['data'].iloc[0]} a {dataset['data'].iloc[-1]}), "
              f"{dataset.shape[1]} colunas.")
linhas.append(f"- Dias sem valor válido na origem (ausentes, nulos ou inválidos): {fmt(dataset['flag_ausente_original'].sum())}; "
              f"imputados por interpolação: {fmt(dataset['flag_imputado'].sum())}; "
              f"sem valor após tratamento: {fmt(dataset['carga_mwmed'].isna().sum())}.")
linhas.append(f"- Dias sinalizados como atípicos: {fmt(dataset['flag_atipico'].sum())} "
              f"({pctf(100*dataset['flag_atipico'].mean())}%), mantidos na base.")

linhas.append("\n## Rastreabilidade")
linhas.append(f"- Hash SHA-256 e data de obtenção de cada arquivo em `registro_downloads.csv`.")
linhas.append(f"- Ambiente: Python {platform.python_version()}, pandas {pd.__version__}, numpy {np.__version__}.")
linhas.append(f"- Parâmetros: janela a partir de {JANELA_INICIO}; lacuna máxima imputável {MAX_LACUNA_IMPUTAVEL} dias; "
              f"limiar de outlier |z| > {LIMIAR_Z_ROBUSTO}.")

resumo = "\n".join(linhas)
(OUT / "resumo_resultados.md").write_text(resumo, encoding="utf-8")

parametros = {"ANO_INICIO_INVENTARIO": ANO_INICIO_INVENTARIO, "ANO_FIM": ANO_FIM, "SUBSISTEMA_ALVO": SUBSISTEMA_ALVO,
              "JANELA_INICIO": JANELA_INICIO, "MAX_LACUNA_IMPUTAVEL": MAX_LACUNA_IMPUTAVEL,
              "LIMIAR_Z_ROBUSTO": LIMIAR_Z_ROBUSTO, "USAR_INMET": USAR_INMET,
              "data_execucao": DATA_EXECUCAO.isoformat(timespec="seconds"),
              "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__}
(OUT / "parametros_execucao.json").write_text(json.dumps(parametros, indent=2, ensure_ascii=False), encoding="utf-8")

print(resumo)
print("\nArquivos gerados:")
for p in sorted(list(OUT.iterdir()) + list(FIG.iterdir())):
    print(" -", p)

## 8. Baixar todos os arquivos do Colab


In [ ]:
import shutil
pasta_zip = BASE / "_entrega"
if pasta_zip.exists():
    shutil.rmtree(pasta_zip)
pasta_zip.mkdir()
shutil.copytree(OUT, pasta_zip / "saidas")
shutil.copytree(FIG, pasta_zip / "figuras")
arquivo_zip = shutil.make_archive("etapa2_resultados", "zip", pasta_zip)
shutil.rmtree(pasta_zip)
print("Gerado:", arquivo_zip)
try:
    from google.colab import files
    files.download(arquivo_zip)
except ImportError:
    pass